# Checking the emulator against the full model

_Never trust an emulator you haven't tested against the thing it replaces._


In ["DSI basics"](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) we trained a DSI emulator on the prior Monte Carlo ensemble, history-matched it against the synthetic truth in seconds, and read a posterior forecast off the emulator: the distribution of peak SO₄ at the supply well (`wellopt`) over the supply period (days 308–728), summarised as its median and P95. That whole exercise never touched the process model. The emulator stood in for it.

Before we lean on the emulator for dataworth ([part1_07](../part1_07_dataworth/dizon_dataworth.ipynb)) and optimization ([part1_08](../part1_08_optimization/dizon_optimization.ipynb)), we owe it one honest test. We earlier ran a *fidelity check* on the emulator's **prior** predictions (held-out realisations the emulator never saw). That tested whether DSI reproduces the prior. It did not test whether DSI's **posterior** — the result of conditioning the emulator on data — matches what the full reactive-transport model produces when *it* is history-matched against the same data.

So here we do the expensive thing, once. We history-match the full DIZON model with PESTPP-IES — the same algorithm, the same observations, the same noise, the same synthetic truth — and compare its posterior against the DSI posterior on the forecast. If they agree, the emulation shortcut is validated and everything downstream rides on the cheap method with a clear conscience. If they disagree, we need to know *where* and *how much* before we make decisions on the emulator.

This is the notebook where the thesis of the whole series — *emulate, because full-model ensembles are unaffordable for reactive transport* — gets its receipt. The full-model history match cost **201 realisations × 4 iterations × ~6 min/run** of compute. We ran it for you. You do not re-run it.


### Admin

We start with the usual imports and dependency checks. As in the rest of the series, the vendored `flopy` and `pyemu` are pulled from the repo's `dependencies/` tree — the asserts below fail loudly if a system install has shadowed them.


In [ ]:
import os
import sys
import shutil
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0, "..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10


**Prerequisite check.** This notebook consumes the DSI posterior produced in ["DSI basics"](../part1_05_dsi_basics/dizon_dsi_basics.ipynb). It also consumes the prebaked full-model history-match results (see below). If the DSI workspace is missing, go run that notebook first.


In [ ]:
# TODO: point at the DSI conditioning workspace written by part1_05.
# dsi_d = Path(os.path.join('..', 'part1_05_dsi_basics', 'dsi_template'))
# if not (dsi_d / 'dsi.pst').exists():
#     raise Exception("you need to run the '../part1_05_dsi_basics/dizon_dsi_basics.ipynb' notebook")


### What was run for you

The full-model history match is the one genuinely expensive computation in part1. Re-running it on a laptop is a non-starter: each forward run of the DIZON reactive-transport model takes **~6 minutes**, and PESTPP-IES needs the whole ensemble evaluated at every iteration. The bill:

| | |
|---|---|
| Realisations drawn / kept | 201 (the same prior ensemble used for prior MC and DSI training) |
| IES iterations | 4 (prior + 3 update iterations: `noptmax=3`) |
| Cost per forward run | ~6 min |
| Order-of-magnitude wall time | ~201 × 4 × 6 min ≈ **80 hours** of model evaluation |

Parallelised across worker agents this comes down from days to many hours, but it is still the kind of run you launch overnight, not the kind you sit and watch. Contrast that with the DSI conditioning in the previous notebook, which finished in **seconds**. That gap is the entire reason this series is built around emulation.


We have stored the *thinned* results of that run — the parameter and observation ensembles per iteration, restricted to the curated observation set plus the forecast — in the tracked [`prebaked/`](../../prebaked/) directory. The raw run directory (`master_hm`, thousands of model files) is far too large to track; the thinned ensembles are a few MB. We load them, we do not regenerate them.

The cell below resolves the prebaked artifact, with a graceful fallback to a local `master_hm` run directory if you happen to have one (e.g. you are the maintainer who produced it).


In [ ]:
# TODO: resolve the full-model history-match results.
# Preference order: tracked prebaked artifact, then a local master_hm run dir.
#
# prebaked_d = Path(os.path.join('..', '..', 'prebaked'))
# hm_pst_prebaked = prebaked_d / 'full_model_hm' / 'pest.pst'   # exact name per prebaked/ inventory
# local_hm = Path(os.path.join('..', '..', 'master_hm', 'pest.pst'))
#
# if hm_pst_prebaked.exists():
#     hm_pst_path = hm_pst_prebaked
#     print('loading prebaked full-model history match')
# elif local_hm.exists():
#     hm_pst_path = local_hm
#     print('falling back to local master_hm run directory')
# else:
#     raise Exception(
#         "could not find the full-model history match. Expected the prebaked artifact at "
#         + str(hm_pst_prebaked) + " (see the prebaked/ inventory in docs/REDESIGN.md)."
#     )


In [ ]:
# TODO: load the full-model control file and its IES ensembles.
# The .pst handle gives us iteration-indexed obs ensembles via the .ies accessor.
#
# pst_fom = pyemu.Pst(str(hm_pst_path))
# oe_fom = pst_fom.ies.obsen          # MultiIndex: (iteration, realization)
# proe_fom = oe_fom.loc[0].copy()     # iteration 0  -> prior
# ptoe_fom = oe_fom.loc[3].copy()     # iteration 3  -> full-model posterior
# print('full-model prior reals:', proe_fom.shape[0], ' posterior reals:', ptoe_fom.shape[0])


A note on what "201 realisations" means in practice. We *drew* and *kept* 201 from the prior, but reactive-transport runs occasionally fail to converge, and IES rejects realisations whose phi blows up. So the prior and posterior ensembles you load may hold slightly fewer than 201 — inspect the printed counts. A handful of dropped realisations is normal and not a cause for alarm; a large fraction missing would be.


### Load the DSI posterior

Now the cheap side of the comparison: the emulator's posterior from [part1_05](../part1_05_dsi_basics/dizon_dsi_basics.ipynb). Same convention — iteration 0 is the (emulated) prior, iteration 3 is the conditioned posterior. We only need the posterior here; the full-model prior is our shared baseline.


In [ ]:
# TODO: load the DSI conditioning results written by part1_05.
#
# pst_dsi = pyemu.Pst(str(dsi_d / 'dsi.pst'))
# oe_dsi = pst_dsi.ies.obsen
# oe3_dsi = oe_dsi.loc[3].copy()      # DSI posterior (iteration 3)
# print('DSI posterior reals:', oe3_dsi.shape[0])
#
# Note: DSI runs are essentially free, so this ensemble is typically much larger than the
# full-model one (e.g. 1000 reals vs ~201). That is fine for comparing distributions; just be
# aware the two ensembles are not the same size when you read the figures.


### Pin down the forecast

The forecast the whole series is judged on is **peak SO₄ at the supply well (`wellopt`) over the supply period** (days 308–728). The canonical definition is the maximum over *all three* supply-well screens (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`) and over all supply-period times, of the `so4` concentration — not a single screen. SO₄ is carried in **mol/L** in the model output, so we convert each realisation's peak to **mg/L** (SO₄ molar mass ≈ 96.06 g/mol), the same decision-readable convention as [part1_05](../part1_05_dsi_basics/dizon_dsi_basics.ipynb). The payoff we compare is the forecast *distribution* — median and P95 — not an exceedance count.

In [ ]:
# TODO: select the supply-well SO4 observation columns over the supply period.
# the canonical forecast maxes over ALL supply-well screens (welopt-ly1/ly3/ly5);
# the spelling in the artifact is 'welopt-...' (CONTEXT.md flags the wellopt/welopt
# drift -- use whatever the loaded pst actually contains).
#
# obs = pst_fom.observation_data.copy()
# obs['time'] = obs['time'].astype(float)
# fore_obs = obs.loc[
#     (obs.obsid.isin(['welopt-ly1', 'welopt-ly3', 'welopt-ly5']))  # all supply-well screens
#     & (obs.variable == 'so4')
#     & (obs.time >= 308) & (obs.time <= 728)
# ]
# fore_cols = fore_obs.obsnme.tolist()
# print('supply-well SO4 forecast columns (all screens):', len(fore_cols))
#
# SO4_MW = 96.06           # g/mol  (mol/L -> mg/L)

In [ ]:
# TODO: reduce each realisation to a single 'peak SO4' value (mg/L) for each ensemble.
# Peak = max over supply-period times across ALL supply-well screens, per realisation,
# converted from mol/L to mg/L (the canon; same convention as part1_05).
#
# def peak_so4_mgl(oe):
#     cols = [c for c in fore_cols if c in oe.columns]
#     return oe.loc[:, cols].max(axis=1) * 1000.0 * SO4_MW
#
# peak_prior = peak_so4_mgl(proe_fom)     # shared baseline (full-model prior)
# peak_fom   = peak_so4_mgl(ptoe_fom)     # full-model posterior
# peak_dsi   = peak_so4_mgl(oe3_dsi)      # DSI posterior
#
# And the truth value of the forecast (synthetic truth realisation, selected in part1_05):
# truth_peak = ...  # carried through from the truth realisation, in mg/L

### The comparison that matters

Everything above was bookkeeping. Here is the test. We overlay the forecast distribution three ways:

- the **full-model prior** (grey) — the shared starting point;
- the **full-model posterior** (orange) — the expensive answer, our reference;
- the **DSI posterior** (blue) — the cheap answer we want to trust.

The payoff is the *distribution*, so we read it as a distribution: where the median sits and where the P95 sits (the operator designs treatment capacity to the P95 of peak SO₄). If the blue and orange distributions land their median and P95 in the same place — the same shift up from the prior, the same tightening — then the emulator answers the decision question the same way the full model does, and the shortcut is vindicated. The full-model posterior loaded here should reproduce **median ≈ 92 mg/L, 5–95% ≈ 83–97 mg/L** (the on-disk IES iter-3 result); the DSI posterior should track it.

In [ ]:
# TODO: histogram / KDE of the peak-SO4 forecast (mg/L) for all three ensembles.
#
# fig, ax = plt.subplots(1, 1, figsize=(7, 4))
# bins = np.linspace(min(peak_prior.min(), peak_fom.min(), peak_dsi.min()),
#                    max(peak_prior.max(), peak_fom.max(), peak_dsi.max()), 40)
# ax.hist(peak_prior, bins=bins, color='0.7', alpha=0.5, density=True, label='full-model prior')
# ax.hist(peak_fom,   bins=bins, color='#ff7f0e', alpha=0.5, density=True, label='full-model posterior')
# ax.hist(peak_dsi,   bins=bins, color='#1f77b4', alpha=0.5, density=True, label='DSI posterior')
# # mark the P95 each posterior would design treatment capacity to
# ax.axvline(peak_fom.quantile(0.95), color='#ff7f0e', ls='--', lw=1.2, label='full-model P95')
# ax.axvline(peak_dsi.quantile(0.95), color='#1f77b4', ls='--', lw=1.2, label='DSI P95')
# # ax.axvline(truth_peak, color='fuchsia', lw=2, label='synthetic truth')
# ax.set_xlabel('peak SO$_4$ at supply well (mg/L)')
# ax.set_ylabel('density')
# ax.legend(fontsize='small')
# fig.tight_layout()

The decision is a design number, so state it as one. For each ensemble report the median and P95 of peak SO₄ (mg/L) — the median is the central forecast, the P95 is what the operator sizes treatment capacity to. Lay them side by side. These are the numbers a decision-maker carries out of the room; if the emulator and the full model disagree on the posterior median or P95, nothing else in the comparison matters.

In [ ]:
# TODO: report the median and P95 of peak SO4 (mg/L) for each ensemble.
#
# def fc_stats(peaks):
#     return [float(peaks.median()), float(peaks.quantile(0.95))]
#
# summary = pd.DataFrame(
#     [fc_stats(peak_prior), fc_stats(peak_fom), fc_stats(peak_dsi)],
#     columns=['median peak SO4 (mg/L)', 'P95 peak SO4 (mg/L)'],
#     index=['full-model prior', 'full-model posterior', 'DSI posterior'],
# )
# summary
#
# Sanity check: the full-model posterior row should read ~92 (median) / ~97 (P95) mg/L,
# the on-disk IES iter-3 values. The DSI posterior should land close to it.

### Where the two posteriors agree — and where they don't

A single forecast number can hide a lot. To see *where* DSI tracks the full model and where it drifts, plot the conditioned breakthrough series at a few diagnostic locations — the supply well itself, plus a couple of monitoring sites that carry strong signal (e.g. `wp1-f3`, `pp1-f3`, `wp4-f5`). For each, overlay the full-model prior (grey), the full-model posterior (orange), the DSI posterior (blue), and the synthetic truth (the magenta line). The history period ends at day 252; the supply period (where the forecast lives) is everything past day 308.


In [ ]:
# TODO: per-location breakthrough plots, prior vs full-model posterior vs DSI posterior vs truth.
# A helper that draws an ensemble as a faint spaghetti band plus its mean keeps this readable.
#
# def plot_band(ax, x, ensemble, color, label, alpha_lines=0.1):
#     arr = np.asarray(ensemble)
#     ax.fill_between(x, np.percentile(arr, 5, axis=0), np.percentile(arr, 95, axis=0),
#                     color=color, alpha=0.1)
#     for line in arr:
#         ax.plot(x, line, color=color, alpha=alpha_lines, lw=0.6)
#     ax.plot(x, arr.mean(axis=0), color=color, lw=2.0, label=label)
#
# obs_to_plot = ['welopt-ly3', 'wp1-f3', 'pp1-f3', 'wp4-f5']   # verified against data/obs_chem_cleaned.csv
# variables   = ['so4', 'ph', 'tmp']
#
# obs = pst_fom.observation_data.copy()
# obs['time'] = obs['time'].astype(float)
# obs['obgnme2'] = obs.obsid + ':' + obs.variable
#
# fig, axs = plt.subplots(len(variables), len(obs_to_plot),
#                         figsize=(3 * len(obs_to_plot), 2.2 * len(variables)),
#                         sharex=True, sharey='row')
# for r, var in enumerate(variables):
#     for c, oid in enumerate(obs_to_plot):
#         ax = axs[r, c]
#         sub = obs.loc[obs.obgnme2 == oid + ':' + var].sort_values('time')
#         if sub.empty:
#             continue
#         t = sub.time.values
#         cols = sub.obsnme.values
#         plot_band(ax, t, proe_fom.loc[:, cols], '0.7',     'prior')
#         plot_band(ax, t, ptoe_fom.loc[:, cols], '#ff7f0e', 'full model')
#         plot_band(ax, t, oe3_dsi.loc[:, cols],  '#1f77b4', 'DSI')
#         # ax.plot(t, truth_series, color='fuchsia', lw=2, label='truth')
#         ax.axvline(252, color='k', ls=':', lw=0.8)   # decision date / end of history
#         ax.set_title(oid + ':' + var, loc='left', fontsize='small')
# fig.tight_layout()


Read these plots honestly. Some things to expect, and to say out loud rather than paper over:

- **In the history period (day ≤ 252)** both posteriors are pulled toward the conditioning data and the truth. If DSI cannot match the full model *here*, where it had data to learn from, the emulator is broken and nothing downstream is safe.
- **In the supply period (day > 308)**, where there is no data, the two posteriors are each extrapolating. This is the honest battleground. DSI emulates the obs-space relationships it saw in the prior ensemble; if the posterior wanders into a corner of parameter space the prior sampled thinly, the emulator's extrapolation can diverge from the full model's.
- **Species that pin the redox front** (SO₄, O₂, NO₃) are where divergence matters most, because the forecast *is* SO₄. Modest disagreement on, say, pH is less consequential than the same disagreement on supply-well SO₄.

If the DSI posterior is systematically narrower than the full-model posterior, the emulator is **over-confident** — it will understate forecast uncertainty and could talk you into a riskier decision than the full model would support. If it is wider, it is conservative — less dangerous, but it leaves dataworth on the table. State which way it leans; do not just assert agreement.


### The verdict

Spell out the conclusion in the terms a decision-maker cares about, not in goodness-of-fit abstractions:

1. Do the DSI and full-model posteriors give the **same forecast distribution** — the same shift in the median, the same P95, to within the precision the design decision needs?
2. If yes: the emulation shortcut is validated for *this* forecast, and parts 1_07–1_08 proceed on the cheap method. The receipt is paid; the rest is free.
3. If no: name the discrepancy, attribute it (prior coverage? a transform artefact? a species the emulator handles poorly?), and decide whether it changes the design. An emulator that is wrong in a direction that does not move the P95 the operator builds to is still useful — but you have to *check*, which is exactly why this notebook exists.

If you prefer a risk-lens read of the same comparison — a supply contract with a 90 mg/L trigger, say — the [DSI basics](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) worked example shows P(peak > 90 mg/L) rising from ≈0.15 (prior) to ≈0.67 (posterior). That is a lens on the same distribution, not a different analysis; the emulator should reproduce it just as it reproduces the median and P95.

And the meta-point, worth repeating because it is the spine of the series: we paid the full-model bill **once**, to validate the emulator. From here on — dataworth, optimization — we never pay it again. Every additional question (would measuring the held-back cations have helped? what is the best supply-well rate and switch-on day?) is answered against the emulator, in seconds, *because* this one comparison earned us the right to trust it.


Next: ["Dataworth"](../part1_07_dataworth/dizon_dataworth.ipynb) — the held-back cations cash in. We retrain the emulator on different observation subsets and watch what each does to the forecast, for nearly nothing, because the prior runs already exist.
